# Validate the PCASP calculation and build its LUT

The nominal geometry follows [Rosenberg et al. (2012), Section 1.2 and Table 1](https://doi.org/10.5194/amt-5-1147-2012): a 632.8 nm laser, one full-azimuth collector at 35–120° relative to the outgoing beam, and a returning beam that sees the same collector at 60–145°. The two beams have equal intensity.

Before building, this notebook compiles the authors' **MieConScat 1.1.8 / Wiscombe MIEV0** source and compares its calculated curves with SizeDistMerge. This is a separate Mie solver, not a comparison with real-aerosol calibration. To compare the same physical quantity, the sum of the author's two beam integrals is divided by two: our cross-section is relative to **total incident irradiance**.

Requirements: the Research Python environment, `gfortran`, and a C++ compiler. Source files are downloaded from the authors' archive, checked against a fixed SHA-256 hash, and compiled in the output directory. Nothing is installed into Python. Run top to bottom. Set environment variable `PCASP_BUILD_LUT=0` for only the literature comparison; the default also builds the LUT. Existing LUTs are never overwritten.

The paper-figure notebook and generated outputs are kept separate from the committed source.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from contextlib import redirect_stdout
from zipfile import ZipFile
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request

for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"
os.environ["MIEPYTHON_USE_JIT"] = "1"

REPO = Path.cwd().resolve()
if not (REPO/"src/sizedistmerge").is_dir():
    REPO = REPO.parent
assert (REPO/"src/sizedistmerge").is_dir()
sys.path.insert(0, str(REPO/"src"))
import numpy as np
import zarr
from joblib import parallel_config
from sizedistmerge import optical_diameter as od

OUTPUT = REPO/"outputs/pcasp_lut_20260912"
LUT_PATH = OUTPUT/"pcasp_sigma_col_632p8nm.zarr"
BUILD_LUT = os.environ.get("PCASP_BUILD_LUT", "1") != "0"
WORKERS = 6
D_RANGE = (60., 6000., 1000)  # Extends beyond nominal 100–3000 nm coverage.
N_RANGE = (1.30, 1.80, .0005)
K_VALUES = np.array([
    0., .0001, .000136, .000186, .000253, .000345, .000471, .000642,
    .000875, .001, .001193, .001627, .002218, .003023, .004122, .005619,
    .00766, .010443, .014237, .01941, .026461, .036074, .04918, .067046,
    .091404, .12461, .16988, .231597, .315735, .430439, .586814, .8,
])
GEOMETRY = od.PCASPGeom(ring_step_deg=.25, reflected_beam_ratio=1.)
SETUP = od.pcasp_optical_setup(GEOMETRY)
CHUNKS = (128, 64, 1)
SOURCE_URL = "https://downloads.sourceforge.net/project/mieconscat/MieConScat_1.1.8.zip"
SOURCE_SHA256 = "b90749e5c8445d897ef1490eac723673904a85bb98213cd5b7d06a10552106d1"
# Agreement tolerance is 0.1%; actual differences are reported, not hidden by this gate.
MAX_REFERENCE_RELATIVE_ERROR = .001

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def code_hashes():
    return {str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in (REPO/"src/sizedistmerge/optical_diameter.py",
                      REPO/"src/sizedistmerge/optical_geometry.py")}

OUTPUT.mkdir(parents=True, exist_ok=True)
if BUILD_LUT:
    assert not LUT_PATH.exists(), f"Choose a new output directory: {LUT_PATH}"
    assert not (OUTPUT/"run_settings.json").exists(), "An earlier build owns this directory."
assert WORKERS <= (os.cpu_count() or 1)
assert od.OPTICAL_MODEL_VERSION == "solid-angle-polarized-cones-v1"
assert SETUP.wavelength_nm == 632.8


## Independent literature calculation

Only the four named source files are extracted, not the archive's installation scripts or binaries. The small driver below calls the authors' unchanged `scatteringcs` function for each published angular interval. Its units are µm² when diameter and wavelength are in µm. It uses 501 angular samples per interval and single-precision Wiscombe amplitudes; SizeDistMerge uses a 0.25° grid and double-precision miepython amplitudes. Small numerical differences are therefore expected.

The 1,687 comparisons span 60–6000 nm and seven refractive indices, including non-absorbing and absorbing spheres. These are explicit sensitivity inputs, not claims about the precise material indices in Rosenberg Figure 1.


In [ ]:
archive = OUTPUT/"MieConScat_1.1.8.zip"
if not archive.exists():
    # Reuse a previously downloaded, hash-checked archive when available.
    local_archive = Path("/tmp/sizedistmerge-MieConScat_1.1.8.zip")
    if local_archive.exists():
        shutil.copyfile(local_archive, archive)
    else:
        urllib.request.urlretrieve(SOURCE_URL, archive)
assert hashlib.sha256(archive.read_bytes()).hexdigest() == SOURCE_SHA256

reference_dir = OUTPUT/"reference_source"
reference_dir.mkdir(exist_ok=True)
members = (
    "MieConScat/Source/MieV0_C/MieV0_C.h",
    "MieConScat/Source/MieV0_C/MieV0_C.cpp",
    "MieConScat/Source/miev0/MIEV0.f",
    "MieConScat/Source/miev0/ErrPack.f",
)
with ZipFile(archive) as source:
    for member in members:
        (reference_dir/Path(member).name).write_bytes(source.read(member))

driver = r'''#include "MieV0_C.h"
#include <iostream>
#include <iomanip>
int main() {
    const double pi = 3.14159265358979323846;
    double diameter_nm, n, k;
    std::cout << std::setprecision(16);
    while (std::cin >> diameter_nm >> n >> k) {
        double outgoing = scatteringcs(n, k, diameter_nm/1000., .6328,
                                       35.*pi/180., 120.*pi/180.);
        double returning = scatteringcs(n, k, diameter_nm/1000., .6328,
                                        60.*pi/180., 145.*pi/180.);
        std::cout << outgoing << " " << returning << "\n";
    }
}
'''
(reference_dir/"reference_driver.cpp").write_text(driver)
fc = shutil.which("gfortran")
cxx = shutil.which("clang++") or shutil.which("g++")
assert fc and cxx, "Install a Fortran compiler and a C++ compiler for this independent check."
commands = [
    [fc, "-O2", "-std=legacy", "-fallow-argument-mismatch", "-c", "MIEV0.f", "ErrPack.f"],
    [cxx, "-O2", "-c", "MieV0_C.cpp", "reference_driver.cpp"],
    [fc, "MIEV0.o", "ErrPack.o", "MieV0_C.o", "reference_driver.o",
     "-lc++" if sys.platform == "darwin" else "-lstdc++", "-o", "reference_pcasp"],
]
compile_log = []
for command in commands:
    result = subprocess.run(command, cwd=reference_dir, capture_output=True, text=True)
    compile_log.append(" ".join(command)+"\n"+result.stdout+result.stderr)
    if result.returncode:
        raise RuntimeError(compile_log[-1])
(OUTPUT/"reference_compile.log").write_text("\n".join(compile_log))
print("Compiled the authors' original MIEV0 and scatteringcs routines.")


In [ ]:
test_D = np.geomspace(60., 6000., 241)
test_ri = np.array([1.3+0j, 1.45+0j, 1.58+0j, 1.8+0j,
                    1.58+.001j, 1.8+.1j, 1.8+.8j])
cases = [(d, m) for m in test_ri for d in test_D]
input_text = "".join(f"{d:.16g} {m.real:.16g} {m.imag:.16g}\n" for d, m in cases)
result = subprocess.run([str(reference_dir/"reference_pcasp")], input=input_text,
                        capture_output=True, text=True, check=True)
(OUTPUT/"reference_solver_output.txt").write_text(result.stdout)
from io import StringIO
beam_integrals = np.loadtxt(StringIO(result.stdout)).reshape(len(test_ri), len(test_D), 2)
# Rosenberg's two integrals are relative to one beam; our denominator includes both.
reference = beam_integrals.mean(axis=2)
ours = np.array([od.pcasp_csca(test_D, m, geom=GEOMETRY) for m in test_ri])
error = np.abs(ours/reference-1)
assert np.all(np.isfinite(reference) & (reference > 0))
assert np.all(np.isfinite(ours) & (ours > 0))
worst = np.unravel_index(np.argmax(error), error.shape)
report = {
    "checked_utc": utc_now(), "source_url": SOURCE_URL,
    "source_archive_sha256": SOURCE_SHA256, "code_sha256": code_hashes(),
    "cases": int(error.size), "diameter_range_nm": [60., 6000.],
    "indices": [[m.real, m.imag] for m in test_ri],
    "normalization": "mean of outgoing and returning integrals; total incident irradiance",
    "max_relative_error": float(error.max()),
    "median_relative_error": float(np.median(error)),
    "p95_relative_error": float(np.percentile(error, 95)),
    "max_relative_error_by_index": error.max(axis=1).tolist(),
    "worst_diameter_nm": float(test_D[worst[1]]),
    "worst_index": [float(test_ri[worst[0]].real), float(test_ri[worst[0]].imag)],
    "acceptance_limit": MAX_REFERENCE_RELATIVE_ERROR,
    "passed": bool(error.max() < MAX_REFERENCE_RELATIVE_ERROR),
    "scope": "published nominal theoretical calculation, not real-aerosol calibration",
}
(OUTPUT/"literature_comparison.json").write_text(json.dumps(report, indent=2)+"\n")
np.savez_compressed(OUTPUT/"literature_comparison.npz", diameter_nm=test_D, indices=test_ri,
                    reference_sigma=reference, sigma=ours, relative_error=error)
print(f"{error.size} cases: maximum difference {100*error.max():.5f}%; "
      f"median {100*np.median(error):.5f}%.")
assert report["passed"], "Literature comparison failed: do not build the LUT."


## Build and verify the LUT

Six worker processes calculate the explicit grid above: 1,000 diameters × 1,001 real-index values × 32 imaginary-index values. Progress is recorded in `build.log`; `build_status.json` distinguishes running, failed, and verified. The optical setup, package versions, Git commit, and hashes of both optical source files are saved. A test-only run does not reserve a build directory.

This does not replace any POPS/UHSAS LUT, select a campaign calibration index, or start aerosol merging. The raw table preserves Mie oscillations; monotone conversion curves are constructed separately when needed.


In [ ]:
if BUILD_LUT:
    source_hashes = code_hashes()
    assert source_hashes == report["code_sha256"]
    settings = {
        "started_utc": utc_now(), "git_commit": subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
        "code_sha256": source_hashes, "python": sys.executable,
        "packages": {name: version(name) for name in ("numpy", "miepython", "zarr", "joblib")},
        "D_range": D_RANGE, "n_range": N_RANGE, "k_values": K_VALUES.tolist(),
        "chunks": CHUNKS, "workers": WORKERS, "parallel_backend": "loky processes",
        "optical_setup": SETUP.to_dict(), "lut_path": str(LUT_PATH),
        "literature_max_relative_error": report["max_relative_error"],
    }
    with (OUTPUT/"run_settings.json").open("x") as f:
        json.dump(settings, f, indent=2)
    status = {"state": "running", "pid": os.getpid(), "started_utc": utc_now(),
              "lut_path": str(LUT_PATH)}
    def save_status():
        status["updated_utc"] = utc_now()
        temp = OUTPUT/"build_status.tmp"
        temp.write_text(json.dumps(status, indent=2)+"\n")
        temp.replace(OUTPUT/"build_status.json")
    save_status()
    try:
        with (OUTPUT/"build.log").open("x") as log, redirect_stdout(log):
            with parallel_config(backend="loky", inner_max_num_threads=1):
                od.build_pcasp_sigma_lut(
                    str(LUT_PATH), GEOMETRY, D_range=D_RANGE, n_range=N_RANGE,
                    k_values=K_VALUES, chunks=CHUNKS, jobs_per_k=WORKERS,
                    parallel_backend="processes")
        root = zarr.open_group(LUT_PATH, mode="r")
        assert root.attrs["build_complete"] is True
        assert root.attrs["instrument"] == "PCASP"
        assert root.attrs["outgoing_irradiance_basis_multiplier"] == 2
        assert od.optical_setup_from_lut_metadata(root.attrs) == SETUP
        d, n, k = [root[f"coords/{key}"][:] for key in ("D_nm", "n", "k")]
        np.testing.assert_allclose(d, np.geomspace(*D_RANGE), rtol=1e-12)
        np.testing.assert_allclose(n, np.arange(N_RANGE[0], N_RANGE[1]+1e-12, N_RANGE[2]))
        np.testing.assert_array_equal(k, K_VALUES)
        sigma = root["sigma_col"]
        assert sigma.shape == (len(d), len(n), len(k))
        for ik in range(len(k)):
            values = sigma[:, :, ik]
            assert np.all(np.isfinite(values) & (values > 0)), f"Invalid values at k index {ik}"
        di = [0, 249, 499, 749, 999]
        table_error = 0.
        for target_n in (1.30, 1.58, 1.80):
            ni = int(np.argmin(abs(n-target_n)))
            for target_k in (0., .001, .8):
                ki = int(np.argmin(abs(k-target_k)))
                direct = od.pcasp_csca(d[di], complex(n[ni], k[ki]), geom=GEOMETRY)
                saved = np.asarray([sigma[j, ni, ki] for j in di])
                np.testing.assert_allclose(saved, direct, rtol=1e-6, atol=0.)
                table_error = max(table_error, float(np.max(abs(saved/direct-1))))
        assert source_hashes == code_hashes(), "Optical source changed during the build."
        status.update(state="verified", shape=list(sigma.shape),
                      max_saved_relative_error=table_error, finished_utc=utc_now())
        save_status()
        print(f"Verified PCASP LUT: {LUT_PATH}")
    except BaseException as exc:
        status.update(state="failed", error=repr(exc))
        save_status()
        raise
else:
    print("Literature comparison only; full LUT build not requested in this execution.")
